In [1]:
# -*- coding: utf-8 -*-
"""
Created on Fri Apr 17 13:46:32 2020

@author: jlbus
"""

'\nCreated on Fri Apr 17 13:46:32 2020\n\n@author: jlbus\n'

In [2]:
import numpy as np
import pandas as pd

from astropy.table import Table
from astropy.io.ascii import read
from astropy.io import ascii

from astropy import units as u
from astropy.coordinates import SkyCoord
from astroquery.gaia import Gaia

Created TAP+ (v1.2.1) - Connection:
	Host: gea.esac.esa.int
	Use HTTPS: True
	Port: 443
	SSL Port: 443
Created TAP+ (v1.2.1) - Connection:
	Host: geadata.esac.esa.int
	Use HTTPS: True
	Port: 443
	SSL Port: 443


In [3]:
def ra2deg(RA):
    hours, minutes,seconds = RA.split(' ',maxsplit=2)
    decimal_hours = float(hours) + float(minutes)/60 + float(seconds)/3600
    ra = decimal_hours*(360/24)
    return(ra)

In [4]:
def dec2deg(DEC):
    degrees,minutes,seconds = DEC.split(' ',maxsplit=2)
    pos_or_neg = degrees[0]
    degrees = degrees.split(pos_or_neg,maxsplit=1)[1]
    
    dec = abs(float(degrees)) + float(minutes)/60 + float(seconds)/3600
    if pos_or_neg == '-':
        dec = -1*dec
    return(dec)

##### 1) create behive coordinate



In [5]:
# got these coordinates from Wikipedia
beehive_ra = '08 40 24'
beehive_dec = '+19 59 00'

# Converted these coordinates to degrees (using functions from HW#1)
# for proper query format (see definitions at bottom of script)
beehive_RAdeg = round(ra2deg(beehive_ra),4) 
beehive_DECdeg = round(dec2deg(beehive_dec),4)

##### 2) cone search with radius 

In [8]:
import time

start = time.clock()

# create the astropy Sky Coordinate object the Gaia query method likes
coord = SkyCoord(ra=beehive_RAdeg, dec = beehive_DECdeg, unit=(u.degree,u.degree))

# the radius of the search is currently 4 degrees
# define radius of query
radius = u.Quantity(4.0, u.deg)

# get the cone search object 
j = Gaia.cone_search_async(coordinate = coord, radius = radius)

# get results from the cone search object
# res will be an AstroPy Table, which is nice!
res = j.get_results()

# check the column names
#res.colnames

end = time.clock()

print ("%.2gs" % (end-start))

/Users/pachiathao/anaconda3/envs/py36/lib/python3.7/site-packages/ipykernel_launcher.py:3: DeprecationWarning: time.clock has been deprecated in Python 3.3 and will be removed from Python 3.8: use time.perf_counter or time.process_time instead
  This is separate from the ipykernel package so we can avoid doing imports until


INFO: Query finished. [astroquery.utils.tap.core]
1.2e+02s


/Users/pachiathao/anaconda3/envs/py36/lib/python3.7/site-packages/ipykernel_launcher.py:22: DeprecationWarning: time.clock has been deprecated in Python 3.3 and will be removed from Python 3.8: use time.perf_counter or time.process_time instead


##### 3) keep important columns and create Pandas DataFrame

In [9]:

# select the columns with data values we're interested in
res_keep = res['source_id','ra','dec','parallax','pmra','pmdec','phot_g_mean_mag','phot_bp_mean_mag','phot_rp_mean_mag','bp_rp','bp_g','g_rp','radial_velocity','teff_val','radius_val','lum_val']

### turn the astropy table into a data frame
### for easy writing to excel

# get colnames of the items we kept, and initialize an empty data frame
gaia_cols = res_keep.colnames
gaia_cone = pd.DataFrame()

# populate the data frame by column
for name in gaia_cols:
    gaia_cone[name] = res_keep[name]


##### 4) Check how many known praesepe stars are in our 4 degree cone

In [12]:
# import praesepe excel file obtained from praesepy_gaia_query.py script
praes_file_name = 'praesepe.xlsx'
praesepe = pd.read_excel(praes_file_name)

# get source id from known praesepe data frame
praes_source_id = praesepe['source_id'].to_numpy()

# get source id from gaia_cone data frame
gaia_source_id = gaia_cone['source_id'].to_numpy()

# find true/false intersection of praesepe sources in gaia sources
src_ID = np.in1d(gaia_source_id, praes_source_id)

number_known_in_cone = sum(src_ID)

print(number_known_in_cone, " confirmed members in cone")

703  confirmed members in cone


##### 5) write cone to excel so we don't have to query Gaia again

In [14]:
# set file location/name for writing to excel, then write to excel
gaia_cone_file_name = 'cone4.xlsx'
gaia_cone.to_excel(gaia_cone_file_name)
print ("File was created")

File was created


##### 5) write cone to excel so we don't have to query Gaia again

### gaia_cone = field_star + praesape 
1) so we need to remove the praesape stars from the gaia cone list so that we have to different list 
2) create a function that will get a randomly choose a number for this certain amount of time
3) create a training set = 2/3 gaia members and 1/3 field stars 
4) combined the list. randomized the order. assigned 1 or 0 if it's a member or not 
5) create a tree regression to make sure that the code works 
6) test it 